# E9 — the beta expansion, tested

Section 9 of the framework closes with a hedge: the small canonical mixer
angle motivates a finite-difference interpretation of the readout, the
observed hierarchy is consistent with that interpretation, and no
experiment varies beta, estimates derivatives at beta = 0, or isolates the
perturbative coefficients. E9 is that experiment. Every simulation runs
through the verbatim `QuIK_circuit` qiskit pipeline with the mixer angle
as the varied parameter; the formula check is arithmetic on qiskit
amplitudes; nothing else simulates anything.

**Four claims under test, each with a hard assert or a measured value.**
1. **Collapse at beta = 0.** The framework says the zero-mixer
   distribution depends only on Hamming weight and is graph-independent
   on the cubic census. Asserted: within-level constancy per graph and
   cross-graph collapse of the sorted vector (prototype spread 6.9e-18).
2. **The finite-difference structure.** Delta_i c(x) = z_i sum over
   neighbors of z_j takes values in {-3, -1, +1, +3} on cubic graphs.
   Asserted exactly in integer arithmetic on every graph.
3. **The first-order coefficient is the Section-15 formula.** The central
   difference [p(+h) - p(-h)] / 2h is asserted against
   D1(y) = sum_i Im[conj(phi(y)) phi(y XOR e_i)] computed from the
   pre-mixer amplitudes (encoder + entangler through qiskit), with
   step-halving asserting the O(h^2) convergence order (prototype: error
   4.4e-6 at h = 1e-3, Richardson ratio 4.00). This assert is the
   replacement for the Section 9 hedge: the readout mechanism becomes a
   verified equation, not a plausible interpretation.
4. **Order assignment of the hierarchy.** The right-derivative of the
   sorted representation at beta = 0+ is the within-Hamming-level sorted
   D1 (levels ordered by their degenerate beta = 0 values; the sorted map
   is not two-sided differentiable at 0, and beta -> 0+ is the physical
   direction toward the canonical 0.1). Ridge probes on those first-order
   features, and on second-central-difference features in the induced
   ordering, answer which hierarchy rungs are decodable at which
   perturbative order. Pre-registered: C3 at first order (the lemma's
   purity-affine-in-tr(A^3) target). The order assignments of C4, C5, C6,
   and diameter are open cells, and the C5 cell is especially live: E7S
   found C5 the most sampling-fragile rung, and a second-order (or
   higher) assignment here would supply its mechanism.

   **Dry-run observation, on record before the full run.** A
   reduced-scope n = 14 pass showed C3 decoding at 1.000 from the
   second-order features and only 0.150 from the first-order features.
   That reading is finer than the pre-registration's phrasing rather
   than contrary to it: purity's graph-dependent term is quadratic in
   D1 (through the sum of D1 squared, an order-beta-squared quantity),
   so C3 can be linearly absent from D1 coordinates while linearly
   present in D2 coordinates, both consistent with the lemma's purity
   mechanism. The full table adjudicates, and the D1 column's own
   content — whatever is linearly readable at strictly first
   order — becomes an open question of independent interest.

**The beta ladder.** Census geometry (median pairwise sorted L1) and the
seven-target R² at beta in {0.0125, 0.025, 0.05, 0.1, 0.2, 0.4, 0.8}.
Pre-registered: log-log slope of the geometry near zero is approximately
one (first-order dominance). The R²(beta) row doubles as the beta half of
the review's parameter-robustness item.

**Scope.** n = 14 in full (509 graphs). The n = 16 census would multiply
the statevector and probe cost several-fold for a mechanism question that
does not need the two-census replication argument, so n = 16 runs as a
seeded 500-graph subset (SEED = 0, drawn without replacement), sized to
match the n = 14 census, with every assert and every table repeated on
it. The pre-registered reading: the n = 16 subset must reproduce the
n = 14 pattern qualitatively — collapse exact, formula assert passing,
same order assignments, slope near one — and the paper sentence is that
the pattern is consistent on a subset of n = 16, with the full-census run
omitted on cost.

**One protocol deviation, cited.** All E9 ridge probes use the top 1000
coordinates of the respective feature vectors rather than full vectors.
E7 places k = 1000 within 0.01 of the full-vector ceiling for every
target at n = 14, and full-vector RidgeCV across 9 feature sets x 7
targets x 2 censuses would add hours for no resolution on an
order-assignment question.

**Runtime.** Thirteen statevectors per graph (seven ladder values, beta =
0, plus-minus h, plus-minus h/2, and the pre-mixer state): about 5
minutes at n = 14 and 30 minutes for the n = 16 subset. The ridge cells
dominate: measured on the n = 14 dry run, the full session projects to
roughly 6 to 7 hours across both graph sets, checkpointed after every
phase and every ladder value, with LADDER_TARGETS as the documented trim
knob if a shorter session is ever needed.

In [1]:
!pip install 'qiskit[visualization]' --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 8.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 60.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.8/9.8 MB 87.2 MB/s eta 0:00:00


## Environment

In [2]:
import pickle
import time

from collections import defaultdict

import numpy as np
import networkx as nx

from scipy.spatial.distance import pdist

from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector

from sklearn.linear_model import RidgeCV
from sklearn.model_selection import KFold
from sklearn.metrics import r2_score

from tqdm.notebook import tqdm


from scipy.linalg import LinAlgWarning
import warnings
warnings.filterwarnings('ignore', category=LinAlgWarning)

In [3]:
SEED = 0
RIDGE_ALPHAS = np.logspace(-14, 2, 17)
NUM_FOLDS = 5

CENSUS_BASES = {
    14: "/kaggle/input/notebooks/lukemiller1987/n14-aaai-dataset/",
    16: "/kaggle/input/notebooks/lukemiller1987/n16-aaai-dataset/",
}
EXPECTED_CUBIC_COUNTS = {14: 509, 16: 4060}
N16_SUBSET_SIZE = 500

# Circuit parameters — canonical, locked; the mixer angle is the varied knob.
MAX_ENCODING_ROTATION = 2.875
ENTANGLER_SCALAR = 2.0
CANONICAL_MIXER = 0.1
REPS = 1

DIFFERENCE_STEP = 1e-3            # h for central differences
BETA_LADDER = (0.0125, 0.025, 0.05, 0.1, 0.2, 0.4, 0.8)
# All seven targets per the approved slate. Trimming to ('C3','C4','C5','C6')
# halves the ladder's ridge cost if a shorter session is ever needed.
LADDER_TARGETS = TARGET_NAMES = ('C3', 'C4', 'C5', 'C6', 'girth',
                                 'diameter', 'spectral_gap')
PROBE_DEPTH = 1000                # E7: within 0.01 of full ceiling at n=14

COLLAPSE_TOLERANCE = 1e-14        # cross-graph sorted-vector spread at beta=0
FORMULA_TOLERANCE = 1e-4          # central difference vs Section-15 formula
RICHARDSON_BOUNDS = (3.0, 5.5)    # h/(h/2) error ratio; exact O(h^2) gives 4

RUN_NS = (14, 16)

## Load, subset, and compute identity-verified targets

n = 14 loads in full; n = 16 is a seeded 500-graph subset drawn without
replacement. The seven targets are recomputed by the sanctioned methods
and the two cubic trace identities are asserted on every graph used.

In [4]:
CENSUS = {}
for num_vertices in RUN_NS:
    with open(CENSUS_BASES[num_vertices]
              + f"n{num_vertices}_data_dict.pkl", 'rb') as census_file:
        census_data = pickle.load(census_file)
    graph_keys = sorted(census_data)
    assert len(graph_keys) == EXPECTED_CUBIC_COUNTS[num_vertices]

    if num_vertices == 16:
        subset_generator = np.random.default_rng(SEED)
        chosen_positions = np.sort(subset_generator.choice(
            len(graph_keys), size=N16_SUBSET_SIZE, replace=False))
        graph_keys = [graph_keys[position] for position in chosen_positions]
        print(f'n=16: seeded subset of {len(graph_keys)} graphs '
              f'(cost citation: the full census multiplies statevector and '
              f'probe cost several-fold for a mechanism question)')

    adjacency_matrices = np.stack(
        [census_data[graph_key]['adj_mat'] for graph_key in graph_keys]
    ).astype(np.int64)

    target_arrays = {name: np.zeros(len(graph_keys)) for name in TARGET_NAMES}
    for graph_position, adjacency_matrix in enumerate(
            tqdm(adjacency_matrices, desc=f'n={num_vertices} targets')):
        graph = nx.from_numpy_array(adjacency_matrix)
        cycle_counter = defaultdict(int)
        for cycle_vertices in nx.simple_cycles(graph, length_bound=6):
            cycle_counter[len(cycle_vertices)] += 1
        shortest_paths = dict(nx.all_pairs_shortest_path_length(graph))
        eccentricities = [max(shortest_paths[source].values())
                          for source in shortest_paths]
        spectrum = np.sort(np.linalg.eigvalsh(
            adjacency_matrix.astype(np.float64)))
        for cycle_length, name in ((3, 'C3'), (4, 'C4'), (5, 'C5'), (6, 'C6')):
            target_arrays[name][graph_position] = cycle_counter[cycle_length]
        target_arrays['girth'][graph_position] = nx.girth(graph)
        target_arrays['diameter'][graph_position] = max(eccentricities)
        target_arrays['spectral_gap'][graph_position] = (spectrum[-1]
                                                         - spectrum[-2])

    cubed_traces = np.trace(adjacency_matrices @ adjacency_matrices
                            @ adjacency_matrices, axis1=1, axis2=2)
    fourth_traces = np.trace(np.stack(
        [np.linalg.matrix_power(adjacency_matrix, 4)
         for adjacency_matrix in adjacency_matrices]), axis1=1, axis2=2)
    assert np.array_equal(cubed_traces,
                          6 * target_arrays['C3'].astype(np.int64))
    assert np.array_equal(fourth_traces,
                          8 * target_arrays['C4'].astype(np.int64)
                          + 15 * num_vertices)

    CENSUS[num_vertices] = {'adjacency_matrices': adjacency_matrices,
                            'target_arrays': target_arrays}
    print(f'n={num_vertices}: {len(graph_keys)} graphs, identities exact')

n=14 targets:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: 509 graphs, identities exact
n=16: seeded subset of 500 graphs (cost citation: the full census multiplies statevector and probe cost several-fold for a mechanism question)


n=16 targets:   0%|          | 0/500 [00:00<?, ?it/s]

n=16: 500 graphs, identities exact


## The circuit — verbatim, with the mixer angle exposed

`QuIK_circuit` is the census producers' class with `mixer_scalar` passed
through, which its constructor already supports. The pre-mixer state is
composed from the class's own encoder and entangler attributes, so the
Section-15 formula is evaluated on amplitudes the class itself built.

In [5]:
class QuIK_circuit:
    """Builds the QuIC circuit for a given adjacency matrix.

    Degree-proportional RX encoder, RZZ entangler over edges, uniform RX
    mixer, (entangler + mixer) repeated `reps` times. Copied verbatim from
    the census dataset producers; E9 varies mixer_scalar.
    """

    def __init__(self, adj_mat,
                 max_encoding_rotation=MAX_ENCODING_ROTATION,
                 entangler_scalar=ENTANGLER_SCALAR,
                 mixer_scalar=CANONICAL_MIXER,
                 encoding_scalar=None,
                 degree_sequence=None,
                 reps=REPS):
        self.adj_mat = np.asarray(adj_mat)

        if degree_sequence is None:
            self.degree_sequence = self.adj_mat.sum(axis=1)
        else:
            if not np.array_equal(self.adj_mat.sum(axis=1), degree_sequence):
                raise ValueError('provided degree_sequence does not match adj_mat')
        self.max_degree = self.adj_mat.sum(axis=1).max()

        self.max_encoding_rotation = max_encoding_rotation
        self.entangler_scalar = entangler_scalar
        self.mixer_scalar = mixer_scalar
        self.reps = reps

        self.num_qubits = self.adj_mat.shape[0]
        self.qubits = list(range(self.num_qubits))
        self.encoding_scalar = (self.max_encoding_rotation / self.max_degree
                                if encoding_scalar is None else encoding_scalar)

        self.quik_circuit = self.build_quik_circuit()

    def build_encoder(self):
        encoder = QuantumCircuit(self.num_qubits)
        for i, degree in enumerate(self.adj_mat.sum(axis=1)):
            encoder.rx(self.encoding_scalar * degree, i, 'Degree Encoder')
        return encoder

    def build_entangler(self):
        entangler = QuantumCircuit(self.num_qubits)
        edge_u, edge_v = np.nonzero(np.triu(self.adj_mat, k=1))
        for u, v in zip(edge_u, edge_v):
            entangler.rzz(self.entangler_scalar, u, v)
        return entangler

    def build_mixer(self):
        mixer = QuantumCircuit(self.num_qubits)
        mixer.rx(self.mixer_scalar, self.qubits, 'Mixer')
        return mixer

    def build_quik_circuit(self):
        self.encoder = self.build_encoder()
        self.mixer = self.build_mixer()
        self.entangler = self.build_entangler()

        circuit = self.encoder.copy()
        for _ in range(self.reps):
            circuit.append(self.entangler, self.qubits)
            circuit.append(self.mixer, self.qubits)
        circuit.measure_all()
        return circuit


def unsorted_probabilities(adjacency_matrix, mixer_angle):
    """Unsorted Born probabilities at the given mixer angle, verbatim path."""
    circuit = QuIK_circuit(adjacency_matrix,
                           mixer_scalar=mixer_angle).quik_circuit
    circuit = circuit.decompose().remove_final_measurements(inplace=False)
    return Statevector.from_instruction(circuit).probabilities()


def premixer_amplitudes(adjacency_matrix):
    """Amplitudes after encoder and entangler, before any mixer."""
    builder = QuIK_circuit(adjacency_matrix, mixer_scalar=0.0)
    circuit = builder.encoder.copy()
    circuit.append(builder.entangler, builder.qubits)
    return np.asarray(Statevector.from_instruction(circuit.decompose()))

## Claims 1 and 2 — collapse at beta = 0 and the cut finite difference

Both asserted on every graph of both sets. The collapse assert also
records the level structure needed later for the block-sorted derivative
features.

In [6]:
HAMMING_WEIGHTS = {}
LEVEL_ORDER_KEYS = {}
for num_vertices in RUN_NS:
    state_count = 1 << num_vertices
    state_indices = np.arange(state_count)
    HAMMING_WEIGHTS[num_vertices] = np.array(
        [bin(state_index).count('1') for state_index in range(state_count)])

    reference_sorted = None
    zero_beta_unsorted = None
    for adjacency_matrix in tqdm(CENSUS[num_vertices]['adjacency_matrices'],
                                 desc=f'n={num_vertices} beta=0 collapse'):
        zero_beta = unsorted_probabilities(adjacency_matrix.astype(np.float64),
                                           0.0)
        for weight in range(num_vertices + 1):
            level_values = zero_beta[HAMMING_WEIGHTS[num_vertices] == weight]
            assert level_values.max() - level_values.min() < 1e-15, (
                f'n={num_vertices}: beta=0 level {weight} not constant')
        sorted_vector = np.sort(zero_beta)[::-1]
        if reference_sorted is None:
            reference_sorted = sorted_vector
            zero_beta_unsorted = zero_beta
        else:
            assert np.abs(sorted_vector
                          - reference_sorted).max() < COLLAPSE_TOLERANCE, (
                f'n={num_vertices}: beta=0 census collapse violated')
    LEVEL_ORDER_KEYS[num_vertices] = zero_beta_unsorted   # graph-independent
    print(f'n={num_vertices}: beta=0 collapse asserted across all graphs '
          f'(distribution depends only on Hamming weight, graph-independent)')

    # Delta_i c in {-3, -1, +1, +3}, exact integers, every graph
    bit_matrix = ((state_indices[:, None]
                   >> np.arange(num_vertices)[None, :]) & 1)
    sign_matrix = 1 - 2 * bit_matrix
    for adjacency_matrix in CENSUS[num_vertices]['adjacency_matrices']:
        cut_differences = sign_matrix * (sign_matrix @ adjacency_matrix)
        assert set(np.unique(cut_differences).tolist()) <= {-3, -1, 1, 3}, (
            f'n={num_vertices}: Delta_i c outside {{-3,-1,1,3}}')
    print(f'n={num_vertices}: Delta_i c in {{-3, -1, +1, +3}} exact on every graph')

n=14 beta=0 collapse:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: beta=0 collapse asserted across all graphs (distribution depends only on Hamming weight, graph-independent)
n=14: Delta_i c in {-3, -1, +1, +3} exact on every graph


n=16 beta=0 collapse:   0%|          | 0/500 [00:00<?, ?it/s]

n=16: beta=0 collapse asserted across all graphs (distribution depends only on Hamming weight, graph-independent)
n=16: Delta_i c in {-3, -1, +1, +3} exact on every graph


## Claim 3 — the first-order coefficient is the Section-15 formula

Per graph: the central difference at h and h/2 against the formula
computed from pre-mixer amplitudes, with the Richardson ratio asserting
the O(h^2) order. The second central difference is collected in the same
pass. The block-sorted derivative features (levels by their beta = 0
values descending, D1 descending within levels, D2 in the induced order)
are built here and truncated to the probe depth.

In [7]:
DERIVATIVE_FEATURES = {}
FORMULA_ERROR_STATS = {}
for num_vertices in RUN_NS:
    state_count = 1 << num_vertices
    state_indices = np.arange(state_count)
    graph_count = len(CENSUS[num_vertices]['adjacency_matrices'])
    first_order_features = np.empty((graph_count, PROBE_DEPTH))
    second_order_features = np.empty((graph_count, PROBE_DEPTH))
    formula_errors, richardson_ratios = [], []

    for graph_position, adjacency_matrix in enumerate(tqdm(
            CENSUS[num_vertices]['adjacency_matrices'],
            desc=f'n={num_vertices} derivatives')):
        adjacency_float = adjacency_matrix.astype(np.float64)
        probability_plus = unsorted_probabilities(adjacency_float,
                                                  +DIFFERENCE_STEP)
        probability_minus = unsorted_probabilities(adjacency_float,
                                                   -DIFFERENCE_STEP)
        probability_zero = unsorted_probabilities(adjacency_float, 0.0)
        first_order_fd = ((probability_plus - probability_minus)
                          / (2 * DIFFERENCE_STEP))
        second_order_fd = ((probability_plus - 2 * probability_zero
                            + probability_minus) / DIFFERENCE_STEP ** 2)

        amplitudes = premixer_amplitudes(adjacency_float)
        first_order_formula = np.zeros(state_count)
        for qubit_index in range(num_vertices):
            flipped_indices = state_indices ^ (1 << qubit_index)
            first_order_formula += np.imag(np.conj(amplitudes)
                                           * amplitudes[flipped_indices])
        error_at_h = np.abs(first_order_fd - first_order_formula).max()
        assert error_at_h < FORMULA_TOLERANCE, (
            f'n={num_vertices} graph {graph_position}: central difference vs '
            f'Section-15 formula differ by {error_at_h:.2e}')

        half_step = DIFFERENCE_STEP / 2
        first_order_fd_half = (
            (unsorted_probabilities(adjacency_float, +half_step)
             - unsorted_probabilities(adjacency_float, -half_step))
            / (2 * half_step))
        error_at_half = np.abs(first_order_fd_half
                               - first_order_formula).max()
        richardson_ratio = error_at_h / error_at_half
        assert RICHARDSON_BOUNDS[0] < richardson_ratio < RICHARDSON_BOUNDS[1], (
            f'n={num_vertices} graph {graph_position}: Richardson ratio '
            f'{richardson_ratio:.2f} outside {RICHARDSON_BOUNDS}')
        formula_errors.append(float(error_at_h))
        richardson_ratios.append(float(richardson_ratio))

        # block-sorted features: the right-derivative of the sorted map at 0+,
        # using the exact formula values for D1 (verified against the fd above)
        block_order = np.lexsort((-first_order_formula,
                                  -LEVEL_ORDER_KEYS[num_vertices]))
        first_order_features[graph_position] = (
            first_order_formula[block_order][:PROBE_DEPTH])
        second_order_features[graph_position] = (
            second_order_fd[block_order][:PROBE_DEPTH])

    DERIVATIVE_FEATURES[num_vertices] = {'D1': first_order_features,
                                         'D2': second_order_features}
    FORMULA_ERROR_STATS[num_vertices] = {
        'max_formula_error': float(np.max(formula_errors)),
        'richardson_ratio_range': (float(np.min(richardson_ratios)),
                                   float(np.max(richardson_ratios)))}
    print(f"n={num_vertices}: formula verified on every graph "
          f"(worst error {FORMULA_ERROR_STATS[num_vertices]['max_formula_error']:.2e}, "
          f"Richardson ratios "
          f"{FORMULA_ERROR_STATS[num_vertices]['richardson_ratio_range']})")

n=14 derivatives:   0%|          | 0/509 [00:00<?, ?it/s]

n=14: formula verified on every graph (worst error 4.37e-06, Richardson ratios (3.9999869176574436, 4.000000376126507))


n=16 derivatives:   0%|          | 0/500 [00:00<?, ?it/s]

n=16: formula verified on every graph (worst error 5.39e-06, Richardson ratios (3.99998923581222, 3.999999329766847))


## Claim 4 — order assignment of the hierarchy

The verbatim ridge protocol on the first-order features, the second-order
features, and their concatenation, for all seven targets, per census.
Splits are KFold(5, shuffle, seed 0) over each graph set.

In [8]:
ORDER_RESULTS = {}
for num_vertices in RUN_NS:
    graph_count = len(CENSUS[num_vertices]['adjacency_matrices'])
    fold_generator = KFold(n_splits=NUM_FOLDS, shuffle=True,
                           random_state=SEED)
    fold_splits = list(fold_generator.split(np.arange(graph_count)))
    CENSUS[num_vertices]['fold_splits'] = fold_splits

    feature_sets = {
        'D1_first_order': DERIVATIVE_FEATURES[num_vertices]['D1'],
        'D2_second_order': DERIVATIVE_FEATURES[num_vertices]['D2']}

    ORDER_RESULTS[num_vertices] = {}
    for feature_name, feature_matrix in feature_sets.items():
        ORDER_RESULTS[num_vertices][feature_name] = {}
        for target_name in tqdm(TARGET_NAMES,
                                desc=f'n={num_vertices} {feature_name}'):
            target_values = CENSUS[num_vertices]['target_arrays'][target_name]
            fold_scores = []
            for train_indices, test_indices in fold_splits:
                probe_model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
                probe_model.fit(feature_matrix[train_indices],
                                target_values[train_indices])
                fold_scores.append(r2_score(
                    target_values[test_indices],
                    probe_model.predict(feature_matrix[test_indices])))
            fold_scores = np.array(fold_scores)
            ORDER_RESULTS[num_vertices][feature_name][target_name] = {
                'r2_mean': float(fold_scores.mean()),
                'r2_sd': float(fold_scores.std())}
    with open('/kaggle/working/e9_beta_expansion.pkl', 'wb') as output_file:
        pickle.dump({'order_results': ORDER_RESULTS,
                     'formula_error_stats': FORMULA_ERROR_STATS}, output_file)
    print(f'n={num_vertices}: order probes complete, checkpointed')

n=14 D1_first_order:   0%|          | 0/7 [00:00<?, ?it/s]

n=14 D2_second_order:   0%|          | 0/7 [00:00<?, ?it/s]

n=14: order probes complete, checkpointed


n=16 D1_first_order:   0%|          | 0/7 [00:00<?, ?it/s]

n=16 D2_second_order:   0%|          | 0/7 [00:00<?, ?it/s]

n=16: order probes complete, checkpointed


## The beta ladder — geometry growth and R²(beta)

Per ladder value: sorted vectors through the pipeline, the census median
pairwise L1, and the seven-target R² at the probe depth. The log-log
slope of the geometry over the three smallest ladder values is the
first-order-dominance measurement.

In [9]:
LADDER_RESULTS = {}
for num_vertices in RUN_NS:
    graph_count = len(CENSUS[num_vertices]['adjacency_matrices'])
    LADDER_RESULTS[num_vertices] = {}
    for mixer_angle in BETA_LADDER:
        sorted_matrix = np.empty((graph_count, 1 << num_vertices))
        for graph_position, adjacency_matrix in enumerate(tqdm(
                CENSUS[num_vertices]['adjacency_matrices'],
                desc=f'n={num_vertices} beta={mixer_angle}')):
            probabilities = unsorted_probabilities(
                adjacency_matrix.astype(np.float64), mixer_angle)
            sorted_matrix[graph_position] = np.sort(probabilities)[::-1]

        median_pairwise_l1 = float(np.median(
            pdist(sorted_matrix, metric='cityblock')))
        truncated_features = sorted_matrix[:, :PROBE_DEPTH]
        target_scores = {}
        for target_name in LADDER_TARGETS:
            target_values = CENSUS[num_vertices]['target_arrays'][target_name]
            fold_scores = []
            for train_indices, test_indices in CENSUS[num_vertices]['fold_splits']:
                probe_model = RidgeCV(alphas=RIDGE_ALPHAS, cv=5)
                probe_model.fit(truncated_features[train_indices],
                                target_values[train_indices])
                fold_scores.append(r2_score(
                    target_values[test_indices],
                    probe_model.predict(truncated_features[test_indices])))
            target_scores[target_name] = {
                'r2_mean': float(np.mean(fold_scores)),
                'r2_sd': float(np.std(fold_scores))}
        LADDER_RESULTS[num_vertices][mixer_angle] = {
            'median_pairwise_l1': median_pairwise_l1,
            'target_scores': target_scores}
        print(f'n={num_vertices} beta={mixer_angle}: median L1 '
              f'{median_pairwise_l1:.3e}')
        with open('/kaggle/working/e9_beta_expansion.pkl', 'wb') as output_file:
            pickle.dump({'order_results': ORDER_RESULTS,
                         'ladder_results': LADDER_RESULTS,
                         'formula_error_stats': FORMULA_ERROR_STATS},
                        output_file)

    small_angles = np.array(BETA_LADDER[:3])
    small_geometry = np.array(
        [LADDER_RESULTS[num_vertices][angle]['median_pairwise_l1']
         for angle in small_angles])
    slope = float(np.polyfit(np.log(small_angles),
                             np.log(small_geometry), 1)[0])
    LADDER_RESULTS[num_vertices]['small_beta_slope'] = slope
    print(f'n={num_vertices}: log-log geometry slope over the three smallest '
          f'beta values = {slope:.3f} (first-order dominance predicts ~1)')

n=14 beta=0.0125:   0%|          | 0/509 [00:00<?, ?it/s]

n=14 beta=0.0125: median L1 1.230e-05


n=14 beta=0.025:   0%|          | 0/509 [00:00<?, ?it/s]

n=14 beta=0.025: median L1 2.398e-05


n=14 beta=0.05:   0%|          | 0/509 [00:00<?, ?it/s]

...

n=16 beta=0.2: median L1 1.726e-04


n=16 beta=0.4:   0%|          | 0/500 [00:00<?, ?it/s]

n=16 beta=0.4: median L1 8.106e-04


n=16 beta=0.8:   0%|          | 0/500 [00:00<?, ?it/s]

n=16 beta=0.8: median L1 2.799e-03
n=16: log-log geometry slope over the three smallest beta values = 0.937 (first-order dominance predicts ~1)


## Summary tables and persistence

In [10]:
for num_vertices in RUN_NS:
    label = ('full census' if num_vertices == 14
             else f'seeded {N16_SUBSET_SIZE}-graph subset')
    print(f'\n=== n={num_vertices} ({label}) — order assignment, R² at '
          f'k={PROBE_DEPTH} ===')
    print(f'{"target":>13} | {"D1 (first)":>12} {"D2 (second)":>12}')
    for target_name in TARGET_NAMES:
        row_cells = '  '.join(
            f"{ORDER_RESULTS[num_vertices][feature_name][target_name]['r2_mean']:+10.3f}"
            for feature_name in ('D1_first_order', 'D2_second_order'))
        print(f'{target_name:>13} | {row_cells}')

    print(f'\n=== n={num_vertices} — beta ladder ===')
    print(f'{"beta":>8} | {"median L1":>10} | '
          + '  '.join(f'{name:>7}' for name in LADDER_TARGETS))
    for mixer_angle in BETA_LADDER:
        ladder_row = LADDER_RESULTS[num_vertices][mixer_angle]
        score_cells = '  '.join(
            f"{ladder_row['target_scores'][name]['r2_mean']:+7.3f}"
            for name in LADDER_TARGETS)
        print(f"{mixer_angle:>8} | {ladder_row['median_pairwise_l1']:>10.3e} "
              f"| {score_cells}")
    print(f"slope (three smallest betas): "
          f"{LADDER_RESULTS[num_vertices]['small_beta_slope']:.3f}")

with open('/kaggle/working/e9_beta_expansion.pkl', 'wb') as output_file:
    pickle.dump({'order_results': ORDER_RESULTS,
                 'ladder_results': LADDER_RESULTS,
                 'formula_error_stats': FORMULA_ERROR_STATS,
                 'config': {'seed': SEED, 'difference_step': DIFFERENCE_STEP,
                            'beta_ladder': BETA_LADDER,
                            'probe_depth': PROBE_DEPTH,
                            'n16_subset_size': N16_SUBSET_SIZE,
                            'run_ns': RUN_NS}}, output_file)
print('\nsaved e9_beta_expansion.pkl')


=== n=14 (full census) — order assignment, R² at k=1000 ===
       target |   D1 (first)  D2 (second)
           C3 |     +0.150      +1.000
           C4 |     +0.204      +0.612
           C5 |     +0.052      +0.180
           C6 |     +0.046      +0.166
        girth |     +0.025      +0.615
     diameter |     -0.008      +0.455
 spectral_gap |     +0.419      +0.615

=== n=14 — beta ladder ===
    beta |  median L1 |      C3       C4       C5       C6    girth  diameter  spectral_gap
  0.0125 |  1.230e-05 |  +1.000   +0.995   +0.856   +0.612   +0.992   +0.554   +0.939
   0.025 |  2.398e-05 |  +1.000   +0.994   +0.789   +0.522   +0.992   +0.539   +0.936
    0.05 |  4.600e-05 |  +1.000   +0.990   +0.832   +0.536   +0.992   +0.537   +0.921
     0.1 |  8.493e-05 |  +1.000   +0.998   +0.928   +0.484   +0.993   +0.548   +0.944
     0.2 |  1.805e-04 |  +1.000   +0.999   +0.954   +0.500   +0.993   +0.547   +0.915
     0.4 |  8.624e-04 |  +1.000   +0.999   +0.971   +0.500   +0.990   +0.5

## Results

(Written after the run. The reading order: (1) the four assert families —
collapse, Delta_i c, formula-versus-difference, Richardson — whose silent
passage is the section's headline: the Section 9 hedge is replaced by
"the first-order readout coefficient was verified against the analytic
formula on every graph of both sets"; (2) the order-assignment table,
read against the pre-registration that C3 is first-order, with the C4,
C5, C6, and diameter rows as the new facts, and the C5 row read jointly
with E7S's sampling-fragility anomaly; (3) the beta ladder: the slope
near one, and the R²(beta) rows as the beta half of parameter
robustness — cells that hold across the ladder are robustness statements,
cells that move are mechanism statements; (4) the n = 16 subset tables,
read only for qualitative agreement with n = 14 per the pre-registered
sentence: the pattern was shown consistent on a subset of n = 16, and the
full-census run is omitted on cost. Verb discipline: the formula is
"verified," the order assignments are "measured," and nothing here
"explains" beyond the isolated coefficients.)